<a href="https://colab.research.google.com/github/Silas-Kamanda/Projects/blob/main/Copy_of_SKW_Swahili_ASR_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" /></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio

print("Installing PEFT, audiomentations, and wandb...")
!pip install peft==0.14.0
!pip install audiomentations==0.36.0
!pip install wandb==0.18.3

try:
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    import torchaudio
    from torchaudio.transforms import SpecAugment
    import transformers
    import datasets
    import accelerate
    import evaluate
    import wandb
    from tqdm.notebook import tqdm
    from sklearn.model_selection import train_test_split
    from datasets import Dataset, DatasetDict
    from transformers import (
        WhisperProcessor,
        WhisperFeatureExtractor,
        WhisperTokenizerFast,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        EarlyStoppingCallback
    )
    from peft import (
        prepare_model_for_kbit_training,
        LoraConfig,
        PeftModel,
        PeftConfig,
        get_peft_model
    )
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union
    import warnings
    import gradio as gr
    import csv
    import soundfile as sf
    import gc
    warnings.filterwarnings("ignore")
except ImportError as e:
    print(f"Import error: {e}")
    raise

print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"WandB version: {wandb.__version__}")
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
    import audiomentations
    print(f"Audiomentations version: {audiomentations.__version__}")
except ImportError as e:
    print(f"Failed to import library: {e}")
    raise ImportError("Required library missing. Check installation logs.")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

In [ ]:
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-qkvout-fc1fc2-aug-9k-samples",
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise+TimeStretch+PitchShift+SpecAugment",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
print(f"CSV found at: {csv_path}")
print(f"Audio base directory: {audio_base_path}")
print(f"Model outputs to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

## Swahili ASR with Whisper + LoRA + Data Augmentation

This notebook fine-tunes OpenAI's Whisper model for Swahili speech recognition using:
- **LoRA** (Low-Rank Adaptation) for parameter-efficient fine-tuning
- **Data Augmentation** (GaussianNoise, TimeStretch, PitchShift, SpecAugment)
- **WandB** for experiment tracking
- **Gradio** interface for inference

Dataset: Mozilla Common Voice Swahili v12.0 (9,000 samples)

Model: openai/whisper-small with LoRA adapters on attention and feed-forward layers